# Projeto C318 - Previsão de Salários no Mercado de IA

**Objetivo**: Desenvolver um modelo preditivo para estimar o salário anual de vagas na área de IA/dados.

**Tipo de Problema**: Regressão

**Variável Alvo**: `salary`

---

## Sumário
1. Configuração e Reprodutibilidade
2. Carregamento dos Dados
3. Análise Exploratória (EDA)
4. Pré-processamento
5. Modelagem Supervisionada
6. Otimização de Hiperparâmetros
7. Modelagem Não Supervisionada
8. Conclusões

---
## 1. Configuração e Reprodutibilidade

In [34]:
# ============================================================
# SEED GLOBAL - Garantir reprodutibilidade
# ============================================================
SEED = 67

import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# Imports
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os
import pandas as pd

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, 
    balanced_accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    classification_report, 
    confusion_matrix,
    ConfusionMatrixDisplay
)

# XGBoost
from xgboost import XGBRegressor

# Configurações
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("Configuração concluída!")
print(f"Seed global: {SEED}")

Configuração concluída!
Seed global: 67


---
## 2. Carregamento dos Dados

In [ ]:
# Carregar dataset
df = pd.read_csv('../data/AI Job Market Dataset.csv')

print(f"Shape: {df.shape}")
print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

In [ ]:
# Visualizar primeiras linhas
df.head(10)

In [ ]:
# Informações do dataset
df.info()

In [ ]:
# Estatísticas descritivas
df.describe()

---
## 3. Análise Exploratória de Dados (EDA)

### 3.1 Verificação de Valores Faltantes

In [ ]:
# Valores faltantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Faltantes': missing,
    'Percentual (%)': missing_pct
})

print("Valores Faltantes por Coluna:")
print(missing_df[missing_df['Faltantes'] > 0] if missing.sum() > 0 else "Nenhum valor faltante!")

### 3.2 Análise da Variável Alvo (salary)

In [ ]:
# Estatísticas do salary
print("Estatísticas do Salário:")
print(f"  Mínimo:  ${df['salary'].min():,.0f}")
print(f"  Máximo:  ${df['salary'].max():,.0f}")
print(f"  Média:   ${df['salary'].mean():,.0f}")
print(f"  Mediana: ${df['salary'].median():,.0f}")
print(f"  Desvio:  ${df['salary'].std():,.0f}")

In [ ]:
# Distribuição do salary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['salary'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Salário (USD)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição do Salário')
axes[0].axvline(df['salary'].mean(), color='red', linestyle='--', label=f"Média: ${df['salary'].mean():,.0f}")
axes[0].axvline(df['salary'].median(), color='green', linestyle='--', label=f"Mediana: ${df['salary'].median():,.0f}")
axes[0].legend()

# Boxplot
axes[1].boxplot(df['salary'], vert=True)
axes[1].set_ylabel('Salário (USD)')
axes[1].set_title('Boxplot do Salário')

plt.tight_layout()
plt.savefig('../reports/salary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Análise das Variáveis Categóricas

In [ ]:
# Variáveis categóricas
cat_cols = ['job_title', 'company_size', 'company_industry', 'country', 
            'remote_type', 'experience_level', 'education_level', 'hiring_urgency']

print("Valores únicos por variável categórica:")
for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

In [ ]:
# Gráficos de barras para variáveis categóricas
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Distribuição: {col}')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Salário por Categoria

In [ ]:
# Salário médio por job_title
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Por cargo
df.groupby('job_title')['salary'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[0, 0], color='steelblue', edgecolor='black')
axes[0, 0].set_title('Salário Médio por Cargo')
axes[0, 0].set_xlabel('Salário (USD)')

# Por país
df.groupby('country')['salary'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[0, 1], color='steelblue', edgecolor='black')
axes[0, 1].set_title('Salário Médio por País')
axes[0, 1].set_xlabel('Salário (USD)')

# Por nível de experiência
df.groupby('experience_level')['salary'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[1, 0], color='steelblue', edgecolor='black')
axes[1, 0].set_title('Salário Médio por Nível de Experiência')
axes[1, 0].set_xlabel('Salário (USD)')

# Por modalidade de trabalho
df.groupby('remote_type')['salary'].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[1, 1], color='steelblue', edgecolor='black')
axes[1, 1].set_title('Salário Médio por Modalidade')
axes[1, 1].set_xlabel('Salário (USD)')

plt.tight_layout()
plt.savefig('../reports/salary_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Análise das Skills

In [ ]:
# Skills binárias
skills_cols = ['skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud']

# Frequência de cada skill
skills_freq = df[skills_cols].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequência
skills_freq.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Frequência de Skills Requeridas')
axes[0].set_xlabel('Número de Vagas')

# Impacto no salário
skill_salary = {}
for skill in skills_cols:
    skill_salary[skill] = df[df[skill] == 1]['salary'].mean()

pd.Series(skill_salary).sort_values(ascending=True).plot(
    kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Salário Médio por Skill')
axes[1].set_xlabel('Salário (USD)')

plt.tight_layout()
plt.savefig('../reports/skills_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.6 Matriz de Correlação

In [ ]:
# Selecionar apenas colunas numéricas
numeric_cols = ['years_experience', 'salary', 'job_openings', 'job_posting_month', 'job_posting_year'] + skills_cols

# Matriz de correlação
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.savefig('../reports/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Pré-processamento

### 4.1 Definição de Variáveis

In [ ]:
# Remover job_id (identificador)
df_model = df.drop(columns=['job_id'])

# Definir X e y
X = df_model.drop(columns=['salary'])
y = pd.qcut(df_model['salary'], q=3, labels=['Baixo', 'Medio', 'Alto'])

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

In [ ]:
# Classificar as variáveis
# Variáveis numéricas (para padronização)
numeric_features = ['years_experience', 'job_posting_month', 'job_posting_year', 'job_openings']

# Variáveis binárias (já são 0/1)
binary_features = ['skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud']

# Variáveis ordinais (têm ordem natural)
ordinal_features = ['experience_level', 'education_level', 'hiring_urgency']

# Variáveis nominais (sem ordem - One-Hot)
nominal_features = ['job_title', 'company_size', 'company_industry', 'country', 'remote_type']

print("Variáveis por tipo:")
print(f"  Numéricas: {numeric_features}")
print(f"  Binárias: {binary_features}")
print(f"  Ordinais: {ordinal_features}")
print(f"  Nominais: {nominal_features}")

### 4.2 Criar Pipeline de Pré-processamento

In [ ]:
# Definir ordens para variáveis ordinais
experience_order = ['Entry', 'Mid', 'Senior']
education_order = ['Bachelor', 'Master', 'PhD']
urgency_order = ['Low', 'Medium', 'High']

# Preprocessador
preprocessor = ColumnTransformer(
    transformers=[
        # Numéricas: StandardScaler
        ('num', StandardScaler(), numeric_features),
        
        # Binárias: manter como estão (passthrough)
        ('bin', 'passthrough', binary_features),
        
        # Ordinais: OrdinalEncoder
        ('ord', OrdinalEncoder(categories=[experience_order, education_order, urgency_order]), 
         ordinal_features),
        
        # Nominais: OneHotEncoder
        ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), 
         nominal_features)
    ],
    remainder='drop'
)

print("Preprocessador criado!")

### 4.3 Divisão dos Dados

In [ ]:
# Split treino/teste (60/20/20)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=SEED, stratify=y
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, stratify=y_temp
)

print(f"Treino:    {X_train.shape[0]:,} amostras ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Validação: {X_validation.shape[0]:,} amostras ({X_validation.shape[0]/len(X)*100:.0f}%)")
print(f"Teste:     {X_test.shape[0]:,} amostras ({X_test.shape[0]/len(X)*100:.0f}%)")

In [ ]:
# Aplicar preprocessamento
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Shape após preprocessamento:")
print(f"  X_train: {X_train_processed.shape}")
print(f"  X_test: {X_test_processed.shape}")

In [ ]:
# Salvar preprocessador
joblib.dump(preprocessor, '../artifacts/preprocessor.joblib')
print("Preprocessador salvo em: artifacts/preprocessor.joblib")

---
## 5. Modelagem Supervisionada

### 5.1 Função de Avaliação

In [28]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Avalia um modelo de regressão e retorna métricas.
    """
    # Previsões
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Métricas de treino
    train_acc = accuracy_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred, average='macro')
    
    # Métricas de teste
    test_acc = accuracy_score(y_test, y_test_pred)
    test_bal_acc = balanced_accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred, average='macro')
    test_recall = recall_score(y_test, y_test_pred, average='macro')
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    
    print(f"\n{'='*50}")
    print(f"Modelo: {model_name}")
    print(f"{'='*50}")
    print(f"\nTREINO:")
    print(f" Acurácia: {train_acc:.4f}")
    print(f" F1-macro: {train_f1:.4f}") # agrega metricas ex: f1-mraco = (0.85 + 0.90) / 2
    print(f"\nTESTE:")
    print(f" Acurácia: {test_acc:.4f}")
    print(f" Acurácia Balanceada: {test_bal_acc:.4f}")
    print(f" Precisão: {test_precision:.4f}")
    print(f" Revocação: {test_recall:.4f}")
    print(f" F1-macro: {test_f1:.4f}")   
    return {
        'model_name': model_name,
        'train_accuracy': train_acc,
        'train_f1_macro': train_f1,
        'test_accuracy': test_acc,
        'test_balanced_accuracy': test_bal_acc,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'test_f1_macro': test_f1
    }

### 5.2 Modelo 1: Baseline (Classe Majoritária)

In [29]:
baseline_model = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline_model.fit(X_train_processed, y_train)

baseline_results = evaluate_model(
    baseline_model, 
    X_train_processed, 
    X_test_processed,  # X_test vem antes
    y_train, 
    y_test, 
    'Baseline (Majoritária)'
)


Modelo: Baseline (Majoritária)

TREINO:
 Acurácia: 0.3333
 F1-macro: 0.1667

TESTE:
 Acurácia: 0.3330
 Acurácia Balanceada: 0.3333
 Precisão: 0.1110
 Revocação: 0.3333
 F1-macro: 0.1665


### 5.3 Modelo 2: Modelo Linear (Logistic Regression)

In [35]:
# Treinar modelo
lr_model = LogisticRegression(random_state=SEED, max_iter=1000)
lr_model.fit(X_train_processed, y_train)

lr_results = evaluate_model(lr_model, X_train_processed, X_test_processed,
                            y_train, y_test, 'Logistic Regression')


Modelo: Logistic Regression

TREINO:
 Acurácia: 0.9406
 F1-macro: 0.9410

TESTE:
 Acurácia: 0.9333
 Acurácia Balanceada: 0.9333
 Precisão: 0.9354
 Revocação: 0.9333
 F1-macro: 0.9338


### 5.4 Modelo 3: Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=SEED,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_results = evaluate_model(rf_model, X_train, y_train, X_test, y_test, "Random Forest")

### 5.4 Modelo 3: XGBoost

In [ ]:
# Treinar modelo
xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=SEED,
    n_jobs=-1
)
xgb_model.fit(X_train_processed, y_train)

# Avaliar
xgb_results = evaluate_model(xgb_model, X_train_processed, X_test_processed, 
                             y_train, y_test, 'XGBoost')

### 5.5 Comparação dos Modelos

In [ ]:
# Criar DataFrame com resultados
results_df = pd.DataFrame([lr_results, rf_results, xgb_results])
results_df = results_df.set_index('model')

print("\nComparação dos Modelos:")
print(results_df.round(2))

In [ ]:
# Visualização da comparação
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# RMSE
results_df[['train_rmse', 'test_rmse']].plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('RMSE (menor é melhor)')
axes[0].set_ylabel('RMSE (USD)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(['Treino', 'Teste'])

# MAE
results_df[['train_mae', 'test_mae']].plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('MAE (menor é melhor)')
axes[1].set_ylabel('MAE (USD)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(['Treino', 'Teste'])

# R²
results_df[['train_r2', 'test_r2']].plot(kind='bar', ax=axes[2], color=['steelblue', 'coral'])
axes[2].set_title('R² (maior é melhor)')
axes[2].set_ylabel('R²')
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend(['Treino', 'Teste'])

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Otimização de Hiperparâmetros

In [ ]:
# GridSearchCV no Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Iniciando GridSearchCV...")
print(f"Total de combinações: {3 * 4 * 3 * 3} = 108")

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=SEED, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_processed, y_train)

print("\nGridSearchCV concluído!")

In [ ]:
# Melhores parâmetros
print("Melhores hiperparâmetros:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nMelhor RMSE (CV): ${-grid_search.best_score_:,.2f}")

In [ ]:
# Avaliar modelo otimizado
best_rf_model = grid_search.best_estimator_

best_rf_results = evaluate_model(best_rf_model, X_train_processed, X_test_processed, 
                                  y_train, y_test, 'Random Forest (Otimizado)')

In [ ]:
# Salvar melhor modelo
joblib.dump(best_rf_model, '../models/best_random_forest.joblib')
print("Modelo salvo em: models/best_random_forest.joblib")

### 6.1 Feature Importance

In [ ]:
# Obter nomes das features após transformação
feature_names = (
    numeric_features + 
    binary_features + 
    ordinal_features + 
    list(preprocessor.named_transformers_['nom'].get_feature_names_out(nominal_features))
)

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': best_rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Top 15 features
plt.figure(figsize=(10, 8))
top_features = importance_df.head(15)
plt.barh(top_features['feature'], top_features['importance'], color='steelblue', edgecolor='black')
plt.xlabel('Importância')
plt.title('Top 15 Features Mais Importantes')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Modelagem Não Supervisionada

### 7.1 PCA - Redução de Dimensionalidade

In [ ]:
# Aplicar PCA
pca = PCA(random_state=SEED)
X_pca_full = pca.fit_transform(X_train_processed)

# Variância explicada
variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variância por componente
axes[0].bar(range(1, len(variance_ratio[:20]) + 1), variance_ratio[:20], 
            color='steelblue', edgecolor='black')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Variância Explicada')
axes[0].set_title('Variância por Componente (Top 20)')

# Variância cumulativa
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 
             marker='o', color='steelblue')
axes[1].axhline(y=0.95, color='red', linestyle='--', label='95% variância')
axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% variância')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Variância Cumulativa')
axes[1].set_title('Variância Cumulativa Explicada')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

# Encontrar número de componentes para 95% da variância
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"\nComponentes para 95% da variância: {n_components_95}")

In [ ]:
# PCA com 2 componentes para visualização
pca_2d = PCA(n_components=2, random_state=SEED)
X_pca_2d = pca_2d.fit_transform(X_train_processed)

# Visualizar com cores por faixa salarial
salary_bins = pd.cut(y_train, bins=3, labels=['Baixo', 'Médio', 'Alto'])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                      c=salary_bins.cat.codes, cmap='viridis', alpha=0.5, s=10)
plt.colorbar(scatter, label='Faixa Salarial (0=Baixo, 1=Médio, 2=Alto)')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variância)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variância)')
plt.title('PCA - Visualização 2D (colorido por faixa salarial)')
plt.tight_layout()
plt.savefig('../reports/pca_2d.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 K-Means Clustering

In [ ]:
# Método do cotovelo para determinar número de clusters
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    kmeans.fit(X_train_processed)
    inertias.append(kmeans.inertia_)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo - K-Means')
plt.xticks(K_range)
plt.grid(True)
plt.tight_layout()
plt.savefig('../reports/kmeans_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Aplicar K-Means com k=4 (baseado no cotovelo)
N_CLUSTERS = 4

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=10)
clusters = kmeans.fit_predict(X_train_processed)

print(f"Clusters encontrados: {N_CLUSTERS}")
print(f"\nDistribuição dos clusters:")
print(pd.Series(clusters).value_counts().sort_index())

In [ ]:
# Visualizar clusters no espaço PCA 2D
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], 
                      c=clusters, cmap='Set1', alpha=0.6, s=15)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} variância)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} variância)')
plt.title('K-Means Clustering (visualização PCA 2D)')
plt.tight_layout()
plt.savefig('../reports/kmeans_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Análise dos clusters - perfil médio
X_train_with_clusters = X_train.copy()
X_train_with_clusters['cluster'] = clusters
X_train_with_clusters['salary'] = y_train.values

# Estatísticas por cluster
print("Perfil médio por cluster:")
print("="*60)

for cluster_id in range(N_CLUSTERS):
    cluster_data = X_train_with_clusters[X_train_with_clusters['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id} ({len(cluster_data):,} vagas):")
    print(f"  Salário médio: ${cluster_data['salary'].mean():,.0f}")
    print(f"  Anos experiência (média): {cluster_data['years_experience'].mean():.1f}")
    print(f"  Cargo mais comum: {cluster_data['job_title'].mode().values[0]}")
    print(f"  País mais comum: {cluster_data['country'].mode().values[0]}")
    print(f"  Nível mais comum: {cluster_data['experience_level'].mode().values[0]}")

In [ ]:
# Boxplot de salário por cluster
plt.figure(figsize=(10, 6))
X_train_with_clusters.boxplot(column='salary', by='cluster', figsize=(10, 6))
plt.title('Distribuição de Salário por Cluster')
plt.suptitle('')
plt.xlabel('Cluster')
plt.ylabel('Salário (USD)')
plt.tight_layout()
plt.savefig('../reports/salary_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Salvar modelo K-Means
joblib.dump(kmeans, '../models/kmeans_model.joblib')
joblib.dump(pca_2d, '../artifacts/pca_2d.joblib')
print("Modelos não supervisionados salvos!")

---
## 8. Conclusões

In [ ]:
# Resumo final dos resultados
print("="*60)
print("RESUMO DO PROJETO")
print("="*60)

print("\n1. DATASET")
print(f"   - Total de registros: {len(df):,}")
print(f"   - Total de features: {X.shape[1]}")
print(f"   - Features após encoding: {X_train_processed.shape[1]}")

print("\n2. MODELAGEM SUPERVISIONADA")
print(f"   Melhor modelo: Random Forest (Otimizado)")
print(f"   - RMSE (teste): ${best_rf_results['test_rmse']:,.2f}")
print(f"   - MAE (teste):  ${best_rf_results['test_mae']:,.2f}")
print(f"   - R² (teste):   {best_rf_results['test_r2']:.4f}")

print("\n3. MODELAGEM NÃO SUPERVISIONADA")
print(f"   - PCA: {n_components_95} componentes para 95% da variância")
print(f"   - K-Means: {N_CLUSTERS} clusters identificados")

print("\n4. ARTEFATOS SALVOS")
print("   - models/best_random_forest.joblib")
print("   - models/kmeans_model.joblib")
print("   - artifacts/preprocessor.joblib")
print("   - artifacts/pca_2d.joblib")
print("   - reports/*.png (gráficos)")

In [ ]:
# Validação cruzada final do melhor modelo
print("\nValidação Cruzada (5-fold) do Melhor Modelo:")
cv_scores = cross_val_score(best_rf_model, X_train_processed, y_train, 
                            cv=5, scoring='neg_root_mean_squared_error')
print(f"  RMSE médio: ${-cv_scores.mean():,.2f} (+/- ${cv_scores.std() * 2:,.2f})")

### Limitações e Próximos Passos

**Limitações identificadas:**
1. Dataset possivelmente sintético (valores muito regulares, anos futuros como 2026)
2. Inconsistências nos dados (Entry-level com 14 anos de experiência)
3. Não há informações sobre benefícios, localização específica ou tipo de contrato

**Possíveis melhorias:**
1. Feature engineering adicional (ex: combinar skills em índice)
2. Testar outros modelos (LightGBM, CatBoost)
3. Análise de resíduos mais detalhada
4. Validar com dados reais do mercado